# Hélio — pipeline Datajud integrado ao projeto

Este notebook coleta os metadados da API pública do Datajud, transforma cada resposta para o contrato do modelo `DatajudRegistro` e salva um Parquet independente para cada ano.

Cada ano será salvo em `data/processed/datajud/tabela_principal_helios_ANO.parquet`, sem sobrescrever o arquivo existente de 2025–2026. A chave da API nunca fica no notebook: use `DATAJUD_API_KEY` no arquivo `.env` local ou como variável de ambiente.

## Fluxo recomendado

1. Execute a célula de configuração.
2. Revise os anos na célula de configuração.
3. Execute a coleta; os arquivos brutos e checkpoints ficam em `data/raw/datajud/`.
4. Execute a consolidação; será criado um arquivo `tabela_principal_helios_ANO.parquet` para cada ano coletado.
5. Valide os Parquets e, se desejar, use `python manage.py importar_datajud --dry-run` para testar o SQLite.
6. A ingestão sem `--arquivo` lerá os Parquets anuais em ordem crescente e, por último, o arquivo atual de 2025–2026.

In [ ]:
from pathlib import Path
import json
import os

def localizar_projeto():
    configurado = os.getenv('HELIO_PROJECT_ROOT')
    if configurado:
        raiz = Path(configurado).expanduser().resolve()
        if (raiz / 'manage.py').is_file():
            return raiz
    atual = Path.cwd().resolve()
    for raiz in (atual, *atual.parents):
        if (raiz / 'manage.py').is_file():
            return raiz
    raise RuntimeError('Execute o notebook dentro do projeto Hélio ou defina HELIO_PROJECT_ROOT.')

def carregar_env(caminho):
    if not caminho.is_file():
        return
    for linha in caminho.read_text(encoding='utf-8').splitlines():
        linha = linha.strip()
        if not linha or linha.startswith('#') or '=' not in linha:
            continue
        chave, valor = linha.split('=', 1)
        chave = chave.strip().lstrip(chr(65279))
        valor = valor.strip().strip(chr(34)).strip(chr(39))
        if not os.environ.get(chave):
            os.environ[chave] = valor

PROJECT_ROOT = localizar_projeto()
carregar_env(PROJECT_ROOT / '.env')

print(f'Projeto: {PROJECT_ROOT}')
print('O notebook não acessa nenhum banco de dados durante a coleta.')

In [ ]:
import csv
import re
import time
from datetime import datetime
import requests

API_KEY = os.getenv('DATAJUD_API_KEY', '').strip()
if not API_KEY:
    raise RuntimeError('DATAJUD_API_KEY não encontrada. Preencha o .env local; não coloque a chave no notebook.')

URL_DATAJUD = os.getenv(
    'DATAJUD_URL',
    'https://api-publica.datajud.cnj.jus.br/api_publica_tjba/_search',
)
ANOS = [int(valor) for valor in os.getenv('DATAJUD_ANOS', ','.join(str(ano) for ano in range(2009, 2025))).split(',') if valor.strip()]
LIMITE_POR_ANO = int(os.getenv('DATAJUD_LIMITE_POR_ANO', '0') or 0) or None
LOTE_INICIAL = 5_000
LOTE_MINIMO = 2_500
LOTE_MAXIMO = 10_000
LOTE_PASSO_SUCESSO = 500
LOTE_PASSO_FALHA = 500
LOTE_PASSO_CONGESTIONAMENTO = 1_000
SUCESSOS_PARA_AUMENTAR = 3
RETRY_INICIAL = 12
RETRY_MINIMO = 9
RETRY_MAXIMO = 15
RETRY_PASSO = 3
TIMEOUT_CONEXAO = 30
TIMEOUT_LEITURA = 300
STATUS_CONGESTIONAMENTO = {429, 502, 503, 504}
CHECKPOINT_VERSION = 7
LOTE = LOTE_INICIAL
ESPERA_RETRY_ATUAL = RETRY_INICIAL
SUCESSOS_CONSECUTIVOS = 0
PASTA_RAW = PROJECT_ROOT / 'data' / 'raw' / 'datajud'
PASTA_PROCESSED = PROJECT_ROOT / 'data' / 'processed' / 'datajud'
ARQUIVO_PARQUET_ATUAL = PASTA_PROCESSED / 'tabela_principal_helios.parquet'
PASTA_RAW.mkdir(parents=True, exist_ok=True)
PASTA_PROCESSED.mkdir(parents=True, exist_ok=True)

def arquivo_parquet_ano(ano):
    return PASTA_PROCESSED / f'tabela_principal_helios_{ano}.parquet'

def localizar_dicionario_municipios():
    configurado = os.getenv('MUNICIPIOS_BAHIA_CSV')
    candidatos = [
        Path(configurado).expanduser() if configurado else None,
        PROJECT_ROOT / 'data' / 'raw' / 'ibge' / 'municipios_bahia.csv',
        Path.home() / 'Downloads' / 'municipios_bahia.csv',
    ]
    for candidato in candidatos:
        if candidato and candidato.is_file():
            return candidato.resolve()
    raise FileNotFoundError('Dicionário de municípios não encontrado. Defina MUNICIPIOS_BAHIA_CSV.')

MUNICIPIOS_CSV = localizar_dicionario_municipios()
H_MAPA_MUNICIPIOS = {}
with MUNICIPIOS_CSV.open('r', encoding='utf-8-sig', newline='') as arquivo:
    leitor = csv.DictReader(arquivo)
    campos = {campo.strip().upper(): campo for campo in leitor.fieldnames or []}
    coluna_codigo = campos.get('CÓDIGO IBGE') or campos.get('CODIGO_IBGE')
    coluna_nome = campos.get('MUNICÍPIO') or campos.get('MUNICIPIO')
    if not coluna_codigo or not coluna_nome:
        raise ValueError(f'Colunas incompatíveis no dicionário: {leitor.fieldnames}')
    for registro in leitor:
        codigo = str(registro.get(coluna_codigo) or '').strip()
        nome = str(registro.get(coluna_nome) or '').strip().upper()
        if re.fullmatch(r'\d{7}', codigo) and nome:
            H_MAPA_MUNICIPIOS[codigo] = nome

TIMEOUT = (30, 300)
CAMPOS_API = [
    'grau', 'numeroProcesso', 'orgaoJulgador.codigo',
    'orgaoJulgador.nome', 'orgaoJulgador.codigoMunicipioIBGE',
    'classe.codigo', 'classe.nome', 'assuntos.codigo', 'assuntos.nome',
]
LIMITES_MODELO = {
    'grau': 2, 'processo': 25, 'id_municipio': 7, 'municipio': 27,
    'nome_classe': 310, 'codigo_orgao': 5, 'nome_orgao': 155,
}

session = requests.Session()
session.headers.update({
    'Authorization': f'ApiKey {API_KEY}',
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Accept-Encoding': 'gzip',
})

def caminho_ano(ano):
    pasta = PASTA_RAW / str(ano)
    pasta.mkdir(parents=True, exist_ok=True)
    return {
        'parcial': pasta / f'tjba_{ano}_raw.jsonl.part',
        'final': pasta / f'tjba_{ano}_raw.jsonl',
        'checkpoint': pasta / f'checkpoint_{ano}_raw.json',
    }

def codigo_numerico(valor):
    texto = str(valor).strip()
    return int(texto) if texto.isdigit() else texto

def json_lista(valores):
    return json.dumps(valores, ensure_ascii=False)

def formatar_processo(valor):
    bruto = str(valor or '').strip()
    if re.fullmatch(r'\d{7}-\d{2}\.\d{4}\.\d\.\d{2}\.\d{4}', bruto):
        return bruto
    if re.fullmatch(r'\d{1,20}', bruto):
        numero = bruto.zfill(20)
        return f'{numero[:7]}-{numero[7:9]}.{numero[9:13]}.{numero[13]}.{numero[14:16]}.{numero[16:20]}'
    if re.fullmatch(r'sigiloso\(.*\)', bruto, flags=re.IGNORECASE):
        return bruto
    raise ValueError(f'Número de processo inválido: {bruto!r}')

def transformar_hit(hit, ano):
    source = hit.get('_source') or {}
    orgao = source.get('orgaoJulgador') or {}
    classe = source.get('classe') or {}
    assuntos = source.get('assuntos') or []
    if isinstance(assuntos, dict):
        assuntos = [assuntos]
    assuntos = [item for item in assuntos if isinstance(item, dict) and item.get('codigo') is not None]
    codigo_municipio = str(orgao.get('codigoMunicipioIBGE') or '').strip()
    codigos_classe = [codigo_numerico(classe['codigo'])] if classe.get('codigo') is not None else []
    nomes_classe = [str(classe.get('nome') or '')] if codigos_classe else []
    codigos_assunto = [codigo_numerico(item['codigo']) for item in assuntos]
    nomes_assunto = [str(item.get('nome') or '') for item in assuntos]
    registro = {
        'ano': str(ano),
        'grau': str(source.get('grau') or ''),
        'processo': formatar_processo(source.get('numeroProcesso')),
        'id_municipio': codigo_municipio,
        'municipio': H_MAPA_MUNICIPIOS.get(codigo_municipio, ''),
        'codigo_classe': json_lista(codigos_classe),
        'nome_classe': json_lista(nomes_classe),
        'codigo_assunto': json_lista(codigos_assunto),
        'nome_assunto': json_lista(nomes_assunto),
        'codigo_orgao': str(orgao.get('codigo') or ''),
        'nome_orgao': str(orgao.get('nome') or ''),
    }
    for campo, limite in LIMITES_MODELO.items():
        if len(registro[campo]) > limite:
            raise ValueError(f'{campo} excede o limite do modelo: {len(registro[campo])} > {limite}')
    return registro

def filtro_ano(ano):
    return {'range': {'dataAjuizamento': {'gte': f'{ano}0101000000', 'lt': f'{ano + 1}0101000000'}}}

def post_retry(payload, contexto):
    global LOTE, ESPERA_RETRY_ATUAL, SUCESSOS_CONSECUTIVOS
    tentativa = 0
    while True:
        tentativa += 1
        lote_usado = LOTE
        requisicao = dict(payload)
        requisicao['size'] = lote_usado
        params = {
            'allow_partial_search_results': 'false',
            'filter_path': 'timed_out,_shards.failed,hits.hits._source,hits.hits.sort',
        }
        erro_curto = None
        congestionamento = False
        try:
            inicio = time.perf_counter()
            resposta = session.post(URL_DATAJUD, json=requisicao, params=params, timeout=(TIMEOUT_CONEXAO, TIMEOUT_LEITURA))
            duracao = time.perf_counter() - inicio
            if resposta.status_code != 200:
                erro_curto = f'HTTP {resposta.status_code}'
                congestionamento = resposta.status_code in STATUS_CONGESTIONAMENTO
            else:
                dados = resposta.json()
                if not isinstance(dados, dict):
                    erro_curto = 'JSON inválido'
                elif dados.get('timed_out'):
                    erro_curto = 'timeout da API'
                    congestionamento = True
                else:
                    shards = dados.get('_shards') or {}
                    falhos = int(shards.get('failed', 0) or 0) if isinstance(shards, dict) else 1
                    hits_obj = dados.get('hits')
                    hits = hits_obj.get('hits') if isinstance(hits_obj, dict) else None
                    if falhos == 0 and not isinstance(hits, list):
                        verificacao = session.post(URL_DATAJUD, json=requisicao, params={'allow_partial_search_results': 'false'}, timeout=(TIMEOUT_CONEXAO, TIMEOUT_LEITURA))
                        if verificacao.status_code == 200:
                            integral = verificacao.json()
                            if integral.get('timed_out'):
                                erro_curto = 'timeout da API'
                                congestionamento = True
                                falhos = 1
                            else:
                                integral_shards = integral.get('_shards') or {}
                                falhos = int(integral_shards.get('failed', 0) or 0)
                                dados = integral
                                hits_obj = integral.get('hits')
                                hits = hits_obj.get('hits') if isinstance(hits_obj, dict) else None
                        else:
                            erro_curto = f'HTTP {verificacao.status_code}'
                            congestionamento = verificacao.status_code in STATUS_CONGESTIONAMENTO
                    if falhos:
                        erro_curto = f'{falhos} shards com falha'
                        congestionamento = True
                    elif not isinstance(hits, list):
                        erro_curto = 'resposta sem hits.hits'
                    elif hits and (not isinstance(hits[-1], dict) or not hits[-1].get('sort')):
                        erro_curto = 'página sem sort'
                    else:
                        ESPERA_RETRY_ATUAL = max(RETRY_MINIMO, ESPERA_RETRY_ATUAL - RETRY_PASSO)
                        SUCESSOS_CONSECUTIVOS += 1
                        if SUCESSOS_CONSECUTIVOS >= SUCESSOS_PARA_AUMENTAR:
                            LOTE = min(LOTE_MAXIMO, LOTE + LOTE_PASSO_SUCESSO)
                            SUCESSOS_CONSECUTIVOS = 0
                        return dados, duracao, lote_usado
        except requests.exceptions.Timeout:
            erro_curto = 'timeout de conexão/leitura'
            congestionamento = True
        except requests.exceptions.RequestException as erro:
            erro_curto = type(erro).__name__
        except (ValueError, TypeError, KeyError) as erro:
            erro_curto = type(erro).__name__
        SUCESSOS_CONSECUTIVOS = 0
        queda = LOTE_PASSO_CONGESTIONAMENTO if congestionamento else LOTE_PASSO_FALHA
        LOTE = max(LOTE_MINIMO, LOTE - queda)
        ESPERA_RETRY_ATUAL = min(RETRY_MAXIMO, ESPERA_RETRY_ATUAL + RETRY_PASSO)
        if tentativa <= 3 or tentativa % 10 == 0:
            print(f'[retry] {contexto} | {erro_curto} | tentativa {tentativa} | lote {LOTE:,} | espera {ESPERA_RETRY_ATUAL}s')
        time.sleep(ESPERA_RETRY_ATUAL)

def consultar_pagina(ano, search_after=None):
    payload = {
        'track_total_hits': False, '_source': CAMPOS_API,
        'query': filtro_ano(ano), 'sort': [{'id.keyword': {'order': 'asc'}}],
    }
    if search_after is not None:
        payload['search_after'] = search_after
    return post_retry(payload, str(ano))

print(f'API configurada: {URL_DATAJUD}')
print(f'Anos configurados: {ANOS}')
print(f'Pasta dos arquivos brutos: {PASTA_RAW}')
print(f'Arquivo existente preservado: {ARQUIVO_PARQUET_ATUAL}')
print(f'Pasta dos Parquets anuais: {PASTA_PROCESSED}')
print(f'Dicionário de municípios: {MUNICIPIOS_CSV} ({len(H_MAPA_MUNICIPIOS):,} entradas)')

In [ ]:
def salvar_checkpoint(caminho, estado):
    temporario = caminho.with_suffix('.tmp')
    with temporario.open('w', encoding='utf-8') as arquivo:
        json.dump(estado, arquivo, ensure_ascii=False, indent=2)
        arquivo.flush()
        os.fsync(arquivo.fileno())
    os.replace(temporario, caminho)

def coletar_ano(ano):
    global LOTE, ESPERA_RETRY_ATUAL, SUCESSOS_CONSECUTIVOS
    caminhos = caminho_ano(ano)
    if caminhos['final'].is_file() and caminhos['final'].stat().st_size > 0:
        print(f'{ano}: arquivo final já existe; coleta não repetida.')
        return caminhos['final']
    parcial = caminhos['parcial']
    checkpoint_path = caminhos['checkpoint']
    if checkpoint_path.is_file():
        estado = json.loads(checkpoint_path.read_text(encoding='utf-8'))
    elif parcial.is_file() and parcial.stat().st_size > 0:
        raise RuntimeError(f'{ano}: existe arquivo parcial sem checkpoint; não vou sobrescrever o progresso.')
    else:
        estado = {'version': CHECKPOINT_VERSION, 'ano': ano, 'search_after': None, 'processos': 0, 'paginas': 0, 'file_size': 0, 'concluido': False}
    if int(estado.get('ano', ano)) != ano or int(estado.get('version', CHECKPOINT_VERSION)) != CHECKPOINT_VERSION:
        raise RuntimeError(f'{ano}: checkpoint incompatível com o ano solicitado ou com a versão atual.')
    tamanho_checkpoint = int(estado.get('file_size', 0))
    tamanho_atual = parcial.stat().st_size if parcial.is_file() else 0
    if tamanho_atual < tamanho_checkpoint:
        raise RuntimeError(f'{ano}: o arquivo parcial é menor que o checkpoint; não é seguro continuar.')
    if tamanho_atual > tamanho_checkpoint:
        with parcial.open('r+b') as arquivo:
            arquivo.truncate(tamanho_checkpoint)
    if estado.get('concluido'):
        if not parcial.is_file():
            raise RuntimeError(f'{ano}: checkpoint concluído sem o arquivo parcial correspondente.')
        parcial.replace(caminhos['final'])
        return caminhos['final']
    total = int(estado.get('processos', 0))
    search_after = estado.get('search_after')
    pagina = int(estado.get('paginas', 0))
    LOTE = LOTE_INICIAL
    ESPERA_RETRY_ATUAL = RETRY_INICIAL
    SUCESSOS_CONSECUTIVOS = 0
    print(f'{ano}: lote adaptativo {LOTE:,} | mínimo {LOTE_MINIMO:,} | máximo {LOTE_MAXIMO:,}')
    concluido = False
    with parcial.open('ab') as arquivo:
        while True:
            dados, duracao, lote_usado = consultar_pagina(ano, search_after)
            hits = (dados.get('hits') or {}).get('hits')
            if not isinstance(hits, list):
                raise RuntimeError(f'{ano}: resposta sem hits.hits após retry.')
            if not hits:
                arquivo.flush()
                os.fsync(arquivo.fileno())
                estado.update({'version': CHECKPOINT_VERSION, 'ano': ano, 'search_after': search_after, 'processos': total, 'paginas': pagina, 'file_size': arquivo.tell(), 'concluido': True})
                salvar_checkpoint(checkpoint_path, estado)
                concluido = True
                break
            novo_search_after = hits[-1].get('sort') if isinstance(hits[-1], dict) else None
            if not novo_search_after or novo_search_after == search_after:
                raise RuntimeError(f'{ano}: search_after não avançou.')
            bloco = ('\n'.join(json.dumps(transformar_hit(hit, ano), ensure_ascii=False, separators=(',', ':')) for hit in hits) + '\n').encode('utf-8')
            arquivo.write(bloco)
            arquivo.flush()
            os.fsync(arquivo.fileno())
            pagina += 1
            total += len(hits)
            search_after = novo_search_after
            estado.update({'version': CHECKPOINT_VERSION, 'ano': ano, 'search_after': search_after, 'processos': total, 'paginas': pagina, 'file_size': arquivo.tell(), 'concluido': False})
            salvar_checkpoint(checkpoint_path, estado)
            print(f'{ano} | página {pagina:,} | recebidos {len(hits):,}/{lote_usado:,} | {duracao:.2f}s | total {total:,} | próximo lote {LOTE:,}')
            if LIMITE_POR_ANO and total >= LIMITE_POR_ANO:
                print(f'{ano}: limite de teste atingido; arquivo parcial preservado.')
                return parcial
    if concluido:
        parcial.replace(caminhos['final'])
        print(f'{ano}: coleta concluída com {total:,} registros.')
        return caminhos['final']

# Coleta todos os anos configurados; por padrão, de 2009 a 2024.
for ano in ANOS:
    coletar_ano(ano)

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

COLUNAS_PARQUET = [
    'ano', 'grau', 'processo', 'id_municipio', 'municipio',
    'codigo_classe', 'nome_classe', 'codigo_assunto', 'nome_assunto',
    'codigo_orgao', 'nome_orgao',
]
SCHEMA_PARQUET = pa.schema([pa.field(coluna, pa.string()) for coluna in COLUNAS_PARQUET])

def normalizar_lista(valor, numerico=False):
    if valor is None:
        valores = []
    elif isinstance(valor, list):
        valores = valor
    elif isinstance(valor, str):
        texto = valor.strip()
        if texto.startswith('[') and texto.endswith(']'):
            try:
                valores = json.loads(texto)
            except json.JSONDecodeError:
                valores = [texto]
            if not isinstance(valores, list):
                valores = [valores]
        else:
            valores = [parte.strip() for parte in texto.split('|')]
    else:
        valores = [valor]
    if numerico:
        return [codigo_numerico(item) for item in valores if str(item).strip()]
    return [str(item).strip() for item in valores if str(item).strip()]

def normalizar_registro(registro, ano):
    id_municipio = str(registro.get('id_municipio') or '').strip()
    municipio = str(registro.get('municipio') or '').strip().upper() or H_MAPA_MUNICIPIOS.get(id_municipio, '')
    normalizado = {
        'ano': str(registro.get('ano') or ano),
        'grau': str(registro.get('grau') or ''),
        'processo': formatar_processo(registro.get('processo')),
        'id_municipio': id_municipio,
        'municipio': municipio,
        'codigo_classe': json_lista(normalizar_lista(registro.get('codigo_classe'), numerico=True)),
        'nome_classe': json_lista(normalizar_lista(registro.get('nome_classe'))),
        'codigo_assunto': json_lista(normalizar_lista(registro.get('codigo_assunto'), numerico=True)),
        'nome_assunto': json_lista(normalizar_lista(registro.get('nome_assunto'))),
        'codigo_orgao': str(registro.get('codigo_orgao') or ''),
        'nome_orgao': str(registro.get('nome_orgao') or ''),
    }
    return {campo: str(normalizado.get(campo) or '') for campo in COLUNAS_PARQUET}

def fontes_coletadas():
    fontes = []
    for ano in ANOS:
        caminhos = caminho_ano(ano)
        if caminhos['final'].is_file():
            fontes.append((ano, caminhos['final'], caminhos['final'].stat().st_size))
        elif caminhos['parcial'].is_file():
            if not caminhos['checkpoint'].is_file():
                raise FileNotFoundError(f'Checkpoint ausente para o ano {ano}.')
            checkpoint = json.loads(caminhos['checkpoint'].read_text(encoding='utf-8'))
            fontes.append((ano, caminhos['parcial'], int(checkpoint['file_size'])))
    if not fontes:
        raise FileNotFoundError('Nenhuma coleta encontrada. Execute coletar_ano primeiro.')
    return fontes

def iterar_registros(caminho, limite):
    with caminho.open('rb') as arquivo:
        while arquivo.tell() < limite:
            linha = arquivo.readline(limite - arquivo.tell())
            if not linha:
                break
            if linha.strip():
                yield json.loads(linha.decode('utf-8'))

def consolidar_parquet_ano(ano, caminho, limite):
    arquivo_final = arquivo_parquet_ano(ano)
    temporario = Path(str(arquivo_final) + '.tmp')
    total = 0
    lote = []
    temporario.unlink(missing_ok=True)
    with pq.ParquetWriter(temporario, SCHEMA_PARQUET, compression='snappy') as escritor:
        for registro in iterar_registros(caminho, limite):
            lote.append(normalizar_registro(registro, ano))
            total += 1
            if len(lote) >= 10_000:
                escritor.write_table(pa.Table.from_pylist(lote, schema=SCHEMA_PARQUET))
                lote.clear()
        if lote:
            escritor.write_table(pa.Table.from_pylist(lote, schema=SCHEMA_PARQUET))
    if total == 0:
        temporario.unlink(missing_ok=True)
        raise RuntimeError(f'Nenhum registro foi consolidado para o ano {ano}.')
    validacao = pq.ParquetFile(temporario)
    linhas_validadas = validacao.metadata.num_rows
    esquema_validado = validacao.schema_arrow
    validacao.close()
    if linhas_validadas != total or esquema_validado != SCHEMA_PARQUET:
        temporario.unlink(missing_ok=True)
        raise RuntimeError(f'A validação do Parquet falhou para o ano {ano}.')
    temporario.replace(arquivo_final)
    print(f'{ano}: Parquet salvo em {arquivo_final} ({total:,} registros; {arquivo_final.stat().st_size / 1024**2:,.2f} MB)')
    return arquivo_final

def consolidar_parquets():
    fontes = fontes_coletadas()
    arquivos = []
    for ano, caminho, limite in fontes:
        arquivos.append(consolidar_parquet_ano(ano, caminho, limite))
    print(f'{len(arquivos)} arquivo(s) anual(is) consolidado(s).')
    return arquivos

# Execute depois que a coleta dos anos desejados terminar.
consolidar_parquets()

## Carga opcional do Parquet no banco local

A população do banco não faz parte da consolidação. Depois de gerar os arquivos, use o comando do projeto; ele descobrirá os Parquets anuais e os importará em ordem crescente. Para importar somente um ano, use `--arquivo`.

In [ ]:
print(f'Parquets anuais gerados em: {PASTA_PROCESSED}')
print(f'Arquivo atual preservado: {ARQUIVO_PARQUET_ATUAL}')
print('O comando sem --arquivo usa os Parquets anuais em ordem crescente:')
print('python manage.py importar_datajud --dry-run')
print('python manage.py importar_datajud --limpar-antes')

In [ ]:
for ano in ANOS:
    arquivo = arquivo_parquet_ano(ano)
    parquet = pq.ParquetFile(arquivo)
    print(f'{ano}: {arquivo} | registros={parquet.metadata.num_rows:,} | grupos={parquet.metadata.num_row_groups}')
    print('Colunas:', parquet.schema_arrow.names)
    amostra = parquet.read_row_group(0, columns=['ano', 'grau', 'id_municipio', 'nome_classe']).slice(0, 5).to_pylist()
    print('Amostra:', amostra)
    parquet.close()

## Configuração no servidor

No servidor, mantenha o mesmo notebook e configure as variáveis de ambiente antes da execução:

```text
DATAJUD_API_KEY=...
DATAJUD_ANOS=2009,2010,...,2024
```

O notebook não imprime a chave, não contém credenciais e salva um arquivo por ano em `data/processed/datajud/`. O comando `python manage.py importar_datajud` importa os arquivos anuais em ordem crescente e depois o arquivo existente de 2025–2026.